# 01. Setup & Cleaning
In questo notebook viene implementata la pipeline iniziale per il caricamento e la pulizia dei dati relativi al mondo degli anime. I dati 'sanificati' vengono poi salvati nella cartella `data/processed/` per le successive fasi di analisi e visualizzazione.

### Obbiettivi
- **Caricamento** dei CSV: importazione dei file necessari per l'analisi.
- **Ispezione e pulizia dei dati**: ricerca di anomalie e rimozione dei duplicati.
- **Gestione valori nulli e tipi di dato**: trattamento dei dati mancanti e correzione dei formati.
- **Salvataggio dei CSV puliti**: Esportazione dei dati pronti all'uso nella cartella `data/processed/`.

In [18]:
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2
from functions.utils import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Caricamento Dataset

In questa sezione carichiamo i file CSV originali per poi visualizzare la dimensione di ogni dataset.

In [2]:
characters_df = pd.read_csv("../data/datasets/characters.csv")
character_anime_works_df = pd.read_csv("../data/datasets/character_anime_works.csv")
character_nicknames_df = pd.read_csv("../data/datasets/character_nicknames.csv")
details_df = pd.read_csv("../data/datasets/details.csv")
favs_df = pd.read_csv("../data/datasets/favs.csv")
person_alternate_names_df = pd.read_csv("../data/datasets/person_alternate_names.csv")
person_anime_works_df = pd.read_csv("../data/datasets/person_anime_works.csv")
person_voice_works_df = pd.read_csv("../data/datasets/person_voice_works.csv")
person_details_df = pd.read_csv("../data/datasets/person_details.csv")
profiles_df = pd.read_csv("../data/datasets/profiles.csv")
ratings_df = pd.read_csv("../data/datasets/ratings.csv")
recommendations_df = pd.read_csv("../data/datasets/recommendations.csv")
stats_df = pd.read_csv("../data/datasets/stats.csv")

In [21]:
show_shape(characters_df, "Characters")
show_shape(character_anime_works_df, "Character_anime_works")
show_shape(character_nicknames_df, "Character_nicknames")
show_shape(details_df, "Details")
show_shape(favs_df, "Favs")
show_shape(person_alternate_names_df, "Person_alternate_names")
show_shape(person_anime_works_df, "Person_anime_works")
show_shape(person_voice_works_df, "Person_voice_works")
show_shape(person_details_df, "Person_details")
show_shape(profiles_df, "Profiles")
show_shape(ratings_df, "Ratings")
show_shape(recommendations_df, "Recommendations")
show_shape(stats_df, "Stats")

Characters: 209506 rows, 7 columns
Character_anime_works: 236816 rows, 4 columns
Character_nicknames: 36962 rows, 2 columns
Details: 28955 rows, 29 columns
Favs: 4178743 rows, 3 columns
Person_alternate_names: 20408 rows, 2 columns
Person_anime_works: 458091 rows, 3 columns
Person_voice_works: 488565 rows, 5 columns
Person_details: 76696 rows, 10 columns
Profiles: 337154 rows, 10 columns
Ratings: 124298350 rows, 6 columns
Recommendations: 105249 rows, 2 columns
Stats: 28955 rows, 27 columns


## Ispezione Dataset

Visualizziamo le prime righe di `Characters`, `Details` e `Profiles` come campioni rappresentativi per capire la struttura generale dei dati.

In [4]:
display(characters_df.head())
display(details_df.head())
display(profiles_df.head())

,character_mal_id,url,name,name_kanji,image,favorites,about
0,280386.0,https://myanimelist.net/character/280386/Envi_...,Envi Mel Champagne,エンヴィ・メル・シャンパーニュ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
1,280354.0,https://myanimelist.net/character/280354/Eleven,Eleven,イレヴン,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
2,280353.0,https://myanimelist.net/character/280353/Stud,Stud,スタッド,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
3,280352.0,https://myanimelist.net/character/280352/Judge,Judge,ジャッジ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
4,280339.0,https://myanimelist.net/character/280339/Eiji_...,Eiji Kurokawa,黒川 英治,https://cdn.myanimelist.net/img/sp/icon/apple-...,0.0,NaN


,mal_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,...,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming
0,59356,-Socket-,-socket-,https://myanimelist.net/anime/59356/-Socket-,https://cdn.myanimelist.net/images/anime/1043/...,Movie,Finished Airing,NaN,NaN,2010-01-01T00:00:00+00:00,...,[],Original,G - All Ages,1.0,NaN,NaN,['Nagoya Zokei University'],[],[],[]
1,56036,......,......,https://myanimelist.net/anime/56036/-,https://cdn.myanimelist.net/images/anime/1057/...,Music,Finished Airing,6.53,503.0,2023-06-11T00:00:00+00:00,...,[],Original,PG-13 - Teens 13 or older,1.0,NaN,NaN,[],[],[],[]
2,2928,.hack//G.U. Returner,.HACK//G.U. RETURNER,https://myanimelist.net/anime/2928/hack__GU_Re...,https://cdn.myanimelist.net/images/anime/1798/...,OVA,Finished Airing,6.65,9745.0,2007-01-18T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,NaN,NaN,"['Bandai Visual', 'CyberConnect2']",[],[],[]
3,3269,.hack//G.U. Trilogy,.hack//G.U. Trilogy,https://myanimelist.net/anime/3269/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/1566/...,Movie,Finished Airing,7.06,15373.0,2007-12-22T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,NaN,NaN,['Bandai Visual'],[],"['Funimation', 'Bandai Entertainment']",[]
4,4469,.hack//G.U. Trilogy: Parody Mode,.hack//G.U. Trilogy,https://myanimelist.net/anime/4469/hack__GU_Tr...,https://cdn.myanimelist.net/images/anime/10/86...,Special,Finished Airing,6.35,4317.0,2008-03-25T00:00:00+00:00,...,[],Game,PG-13 - Teens 13 or older,1.0,NaN,NaN,['Bandai Visual'],[],[],[]


,username,gender,birthday,location,joined,watching,completed,on_hold,dropped,plan_to_watch
0,ishikawas,NaN,NaN,South Korea,NaN,NaN,NaN,NaN,NaN,NaN
1,CKK2,NaN,NaN,United States,"Dec 1, 2018",3,182,15,0,405
2,--------788,Female,NaN,Mexico,"Oct 4, 2022",1,64,0,0,1
3,potatoaris,NaN,NaN,Spain,"Oct 2, 2018",5,1,0,0,4
4,Rinrintan,NaN,NaN,Japan,"May 12, 2019",20,311,40,16,34


## Rimozione dei Duplicati

In questa sezione analizziamo l'integrità dei dati cercando eventuali righe duplicate in tutti i dataset. 

In [ ]:
check_duplicates(characters_df, name="characters")
check_duplicates(character_anime_works_df, name="character_anime_works")
check_duplicates(character_nicknames_df, name="character_nicknames")
check_duplicates(details_df, name="details")
check_duplicates(favs_df, name="favs")
check_duplicates(person_alternate_names_df, name="person_alternate_names")
check_duplicates(person_anime_works_df, name="person_anime_works")
check_duplicates(person_voice_works_df, name="person_voice_works")
check_duplicates(person_details_df, name="person_details")
check_duplicates(profiles_df, name="profiles")
check_duplicates(ratings_df, name="ratings")
check_duplicates(recommendations_df, name="recommendations")
check_duplicates(stats_df, name="stats")

Successivamente, utilizziamo la funzione `clean_duplicates` per rimuoverli definitivamente, garantendo che ogni dataset contenga solo informazioni univoche e che il numero di righe rimosse corrisponda alle copie rilevate prima.

In [5]:
characters_df = clean_duplicates(characters_df, "Characters")
character_nicknames_df = clean_duplicates(character_nicknames_df, "Character_nicknames")
person_alternate_names_df = clean_duplicates(person_alternate_names_df, "Person_alternate_names")
person_voice_works_df = clean_duplicates(person_voice_works_df, "Person_voice_works")
person_details_df = clean_duplicates(person_details_df, "Person_details")

Dataset Characters               : new rows 209507, before was 209963 rows, removed 456
Dataset Character_nicknames      : new rows 36978, before was 37080 rows, removed 102
Dataset Person_alternate_names   : new rows 20426, before was 20465 rows, removed 39
Dataset Person_voice_works       : new rows 488565, before was 489516 rows, removed 951
Dataset Person_details           : new rows 76696, before was 76699 rows, removed 3


## Definizione dei Tipi di Dato (dtypes)

In questa sezione impostiamo il formato corretto per le colonne di ogni dataset.   
Iniziamo visualizzando i tipi di dato di ogni dataset utilizzando l'attributo `.dtypes`.

In [16]:
show_dtypes(characters_df, "Characters")
show_dtypes(character_anime_works_df, "Character_anime_works")
show_dtypes(character_nicknames_df, "Character_nicknames")
show_dtypes(details_df, "Details")
show_dtypes(favs_df, "Favs")
show_dtypes(person_alternate_names_df, "Person_alternate_names")
show_dtypes(person_anime_works_df, "Person_anime_works")
show_dtypes(person_voice_works_df, "Person_voice_works")
show_dtypes(person_details_df, "Person_details")
show_dtypes(profiles_df, "Profiles")
show_dtypes(ratings_df, "Ratings")
show_dtypes(recommendations_df, "Recommendations")
show_dtypes(stats_df, "Stats")


DATASET: Characters
character_mal_id    float64
url                  object
name                 object
name_kanji           object
image                object
favorites             Int64
about                object
dtype: object

DATASET: Character_anime_works
anime_mal_id         int64
character_mal_id     int64
character_name      object
role                object
dtype: object

DATASET: Character_nicknames
character_mal_id     int64
nickname            object
dtype: object

DATASET: Details
mal_id                           int64
title                           object
title_japanese                  object
url                             object
image_url                       object
type                            object
status                          object
score                          float64
scored_by                        Int64
start_date         datetime64[ns, UTC]
end_date           datetime64[ns, UTC]
synopsis                        object
rank                           

Trasformiamo tutte le colonne contenenti date nel formato datetime, utilizzando la funzione `convert_to_datetime` per la gestione dei valori nulli e delle date non valide.

In [ ]:
# Dataset Person_details
person_details_df = convert_to_datetime(person_details_df, "birthday")

# Dataset Profiles
profiles_df = convert_to_datetime(profiles_df, "joined")
profiles_df = convert_to_datetime(profiles_df, "birthday")

# Dataset Details
details_df = convert_to_datetime(details_df, "start_date")
details_df = convert_to_datetime(details_df, "end_date")

c:\Users\ASUS\Desktop\SaraZunino\solution\functions\utils.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column_name] = pd.to_datetime(df[column_name], errors='coerce', format='%Y').fillna(


Column 'birthday' successfully converted.
Column 'joined' successfully converted.
Column 'birthday' successfully converted.
Column 'start_date' successfully converted.
Column 'end_date' successfully converted.


c:\Users\ASUS\Desktop\SaraZunino\solution\functions\utils.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column_name] = pd.to_datetime(df[column_name], errors='coerce', format='%Y').fillna(
c:\Users\ASUS\Desktop\SaraZunino\solution\functions\utils.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[column_name] = pd.to_datetime(df[column_name], errors='coerce', format='%Y').fillna(


Successivamente, convertiamo le restanti colonne numeriche nel tipo più adatto per l'analisi statistica. Nello specifico:
- In `Details`: Le colonne 'years', 'scored_by', 'rank' ed 'episodes' vengono convertite da float a Int64. L'uso di Int64 ci permette di trattare queste colonne come interi pur mantenendo eventuali valori nulli (NaN) senza generare errori.
- In `Profiles`: Le colonne relative allo stato di visione ('watching', 'completed', ecc.) venivano lette come object a causa della virgola usata come separatore delle migliaia. Abbiamo rimosso la virgola per permettere la corretta conversione in Int64.
- In `Characters`: La colonna 'favorites' viene uniformata al tipo Int64.

In [9]:
# Dataset Details
details_df["year"] = details_df["year"].astype("Int64")
details_df["scored_by"] = details_df["scored_by"].astype("Int64")
details_df["rank"] = details_df["rank"].astype("Int64")
details_df["episodes"] = details_df["episodes"].astype("Int64")

# Dataset Characters
characters_df["favorites"] = characters_df["favorites"].astype("Int64")

# Dataset Profiles
if "watching" in profiles_df.columns:
    profiles_df["watching"] = pd.to_numeric(
        profiles_df["watching"].astype(str).str.replace(',', ''), errors='coerce'
    ).astype("Int64")

if "completed" in profiles_df.columns:
    profiles_df["completed"] = pd.to_numeric(
        profiles_df["completed"].astype(str).str.replace(',', ''), errors='coerce'
    ).astype("Int64")

if "on_hold" in profiles_df.columns:
    profiles_df["on_hold"] = pd.to_numeric(
        profiles_df["on_hold"].astype(str).str.replace(',', ''), errors='coerce'
    ).astype("Int64")

if "dropped" in profiles_df.columns:
    profiles_df["dropped"] = pd.to_numeric(
        profiles_df["dropped"].astype(str).str.replace(',', ''), errors='coerce'
    ).astype("Int64")

if "plan_to_watch" in profiles_df.columns:
    profiles_df["plan_to_watch"] = pd.to_numeric(
        profiles_df["plan_to_watch"].astype(str).str.replace(',', ''), errors='coerce'
    ).astype("Int64")

## Analisi e Gestione dei Valori Nulli (NaN)

In questa fase finale della pulizia, ci occupiamo dei valori mancanti all'interno dei nostri dataset.  
Utilizziamo la funzione `check_missing` per visualizzare la percentuale dei valori NaN e identificare in quali colonne specifiche si trovano.

In [15]:
check_missing(characters_df, "Characters")
check_missing(character_anime_works_df, "Character_anime_works")
check_missing(character_nicknames_df, "Character_nicknames")
check_missing(details_df, "Details")
check_missing(favs_df, "Favs")
check_missing(person_alternate_names_df, "Person_alternate_names")
check_missing(person_anime_works_df, "Person_anime_works")
check_missing(person_voice_works_df, "Person_voice_works")
check_missing(person_details_df, "Person_details")
check_missing(profiles_df, "Profiles")
check_missing(ratings_df, "Ratings")
check_missing(recommendations_df, "Recommendations")
check_missing(stats_df, "Stats")

In Characters                col: name_kanji                missing: 26.43%
In Characters                col: about                     missing: 46.19%
In Character_anime_works     No missing values found.
In Character_nicknames       col: nickname                  missing: 0.04%
In Details                   col: title_japanese            missing: 0.41%
In Details                   col: type                      missing: 0.23%
In Details                   col: score                     missing: 34.79%
In Details                   col: scored_by                 missing: 34.79%
In Details                   col: start_date                missing: 2.94%
In Details                   col: end_date                  missing: 61.09%
In Details                   col: synopsis                  missing: 17.71%
In Details                   col: rank                      missing: 24.03%
In Details                   col: rating                    missing: 2.07%
In Details                   col: episo

Attraverso la funzione `drop_empty_rows`, eliminiamo le righe che contengono esclusivamente valori NaN. Queste righe risultano totalmente inutili alla nostra analisi.

In [13]:
characters_df = drop_empty_rows(characters_df, "Characters")
character_anime_works_df = drop_empty_rows(character_anime_works_df, "Character_anime_works")
character_nicknames_df = drop_empty_rows(character_nicknames_df, "Character_nicknames")
details_df = drop_empty_rows(details_df, "Details")
favs_df = drop_empty_rows(favs_df, "Favs")
person_alternate_names_df = drop_empty_rows(person_alternate_names_df, "Person_alternate_names")
person_anime_works_df = drop_empty_rows(person_anime_works_df, "Person_anime_works")
person_voice_works_df = drop_empty_rows(person_voice_works_df, "Person_voice_works")
person_details_df = drop_empty_rows(person_details_df, "Person_details")
profiles_df = drop_empty_rows(profiles_df, "Profiles")
ratings_df = drop_empty_rows(ratings_df, "Ratings")
recommendations_df = drop_empty_rows(recommendations_df, "Recommendations")
stats_df = drop_empty_rows(stats_df, "Stats")

In Characters: removed 1 empty rows


Usiamo la funzione `drop_key_missing` per rimuovere le righe senza le colonne "importanti" in ciascun dataset dove `check_missing` ha rilevato valori mancanti.

In [20]:
# Characters nickname → rimuovi righe senza nickname
character_nicknames_df = drop_key_missing(character_nicknames_df, "nickname", "Character_nicknames")

# Person_alternate_names → rimuovi righe senza alt_name
person_alternate_names_df = drop_key_missing(person_alternate_names_df, "alt_name", "Person_alternate_names")

# Favs → rimuovi righe senza username
favs_df = drop_key_missing(favs_df, "username", "Favs")

# Profiles → rimuovi righe senza username
profiles_df = drop_key_missing(profiles_df, "username", "Profiles")

# Ratings → rimuovi righe senza username
ratings_df = drop_key_missing(ratings_df, "username", "Ratings")


Dataset Character_nicknames: removed 16 rows with NaN in 'nickname'
Dataset Person_alternate_names: removed 18 rows with NaN in 'alt_name'
Dataset Favs           : removed 4 rows with NaN in 'username'
Dataset Profiles       : removed 1 rows with NaN in 'username'
Dataset Ratings        : removed 7 rows with NaN in 'username'


## Salvataggio dei Dataset Puliti

In quest'ultima fase della preparazione, procediamo al salvataggio dei dataset elaborati. Dopo aver rimosso i duplicati, normalizzato i tipi di dato e gestito i valori mancanti, esportiamo i file in una cartella dedicata `data/processed/`.

In [22]:
characters_df.to_csv("../data/processed/Characters_cleaned.csv", index=False)
character_anime_works_df.to_csv("../data/processed/Character_anime_works_cleaned.csv", index=False)
character_nicknames_df.to_csv("../data/processed/Character_nicknames_cleaned.csv", index=False)
details_df.to_csv("../data/processed/Details_cleaned.csv", index=False)
favs_df.to_csv("../data/processed/Favs_cleaned.csv", index=False)
person_alternate_names_df.to_csv("../data/processed/Person_alternate_names_cleaned.csv", index=False)
person_anime_works_df.to_csv("../data/processed/Person_anime_works_cleaned.csv", index=False)
person_voice_works_df.to_csv("../data/processed/Person_voice_works_cleaned.csv", index=False)
person_details_df.to_csv("../data/processed/Person_details_cleaned.csv", index=False)
profiles_df.to_csv("../data/processed/Profiles_cleaned.csv", index=False)
ratings_df.to_csv("../data/processed/Ratings_cleaned.csv", index=False)
recommendations_df.to_csv("../data/processed/Recommendations_cleaned.csv", index=False)
stats_df.to_csv("../data/processed/Stats_cleaned.csv", index=False)